<img src="https://backend.burla.dev/static/welcome.svg" width="45%">

#### This notebook computes **NDVI on 6 public Sentinel-2 tiles** in parallel on Burla, in about 4 minutes. <br/> It's only 4 steps — follow along!

### What is Burla?

Burla is the simplest way to run Python across hundreds or thousands of cloud machines.
One function — `remote_parallel_map` — fans your code out across isolated Docker containers, grows the cluster on demand, and streams results back. Missing packages on the workers? Burla auto-installs them.

No Dockerfiles, no cluster YAML, no queue. Just Python.  
Burla is free to try — this whole notebook runs on machines we spin up for you.

### What will this demo do?

Sentinel-2, Landsat, and NAIP ship as thousands of GeoTIFF tiles. You want NDVI, reprojection, or clipping per tile. A single-core `gdalwarp` loop takes days. GDAL is painful to containerize — native deps, PROJ versions, `GDAL_DATA` paths.

We'll compute **NDVI (vegetation index) on 6 public Sentinel-2 Cloud-Optimized GeoTIFF tiles** hosted on AWS Open Data. Each worker reads just the Red (B04) and NIR (B08) bands it needs (~30 MB of the multi-GB tile thanks to COG range reads), computes `(NIR - Red) / (NIR + Red)`, and returns mean NDVI + pixel count.

The full-scale version in [`main.py`](https://github.com/Burla-Cloud/gdal-raster-processing/blob/main/main.py) processes 2,000 tiles across 2,000 workers.

## 1)&nbsp; Boot some machines (6+ CPUs with 2 GB RAM each):
&nbsp;&nbsp;&nbsp;&nbsp;Sign in using the button below, then hit the **`⏻ Start`** button on your dashboard homepage.  
&nbsp;&nbsp;&nbsp;&nbsp;This will boot a small cluster in Google Cloud for you. Burla is free to try so this is on us!

&nbsp;&nbsp;
<a href="https://login.burla.dev">
    <img src="https://i.ibb.co/QFNncHcp/Clean-Shot-2026-03-19-at-18-13-07.jpg" width="28%">
</a>

## 2)&nbsp; Install the Python package:

In [ ]:
!uv pip install burla

## 3)&nbsp; Authorize this computer:

In [ ]:
!burla login

## 4)&nbsp; Compute NDVI on 6 Sentinel-2 tiles 🚀

The Sentinel-2 Cloud-Optimized GeoTIFFs (COGs) are served over HTTPS from the `sentinel-cogs` AWS Open Data bucket. `rasterio` + `/vsicurl/` streams just the byte ranges it needs — no S3 client, no credentials.

Burla auto-installs `rasterio` on the workers (it bundles GDAL + PROJ), so you don't need a Dockerfile.

In [ ]:
import rasterio  # noqa: F401  # top-level import -> Burla installs rasterio+GDAL on workers
import numpy as np  # noqa: F401
from burla import remote_parallel_map

# 6 public Sentinel-2 tiles from different biomes, August 2023 (low cloud scenes)
BASE = "https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs"
TILES = [
    ("10/T/ET/2023/8/S2B_10TET_20230815_0_L2A", "Pacific Northwest, USA"),
    ("32/T/QS/2023/8/S2B_32TQS_20230815_0_L2A", "Italian Alps"),
    ("18/T/XN/2023/8/S2B_18TXN_20230815_0_L2A", "New York / Hudson Valley"),
    ("22/M/GC/2023/8/S2B_22MGC_20230815_0_L2A", "Amazon Rainforest"),
    ("33/T/UF/2023/8/S2B_33TUF_20230815_0_L2A", "Central Europe"),
    ("56/H/LH/2023/8/S2B_56HLH_20230815_0_L2A", "Sydney, Australia"),
]


def compute_ndvi(job: tuple[str, str]) -> dict:
    import numpy as np
    import rasterio

    tile_id, region = job
    url_red = f"/vsicurl/{BASE}/{tile_id}/B04.tif"
    url_nir = f"/vsicurl/{BASE}/{tile_id}/B08.tif"

    with rasterio.Env(GDAL_HTTP_MAX_RETRY=3, GDAL_HTTP_TIMEOUT=30):
        with rasterio.open(url_red) as r:
            red = r.read(1, out_shape=(1024, 1024)).astype("float32")
        with rasterio.open(url_nir) as r:
            nir = r.read(1, out_shape=(1024, 1024)).astype("float32")

    ndvi = (nir - red) / (nir + red + 1e-6)
    valid = ndvi[(red > 0) & (nir > 0)]

    return {
        "tile_id": tile_id,
        "region": region,
        "pixels": int(valid.size),
        "mean_ndvi": float(valid.mean()) if valid.size else 0.0,
        "p90_ndvi": float(np.percentile(valid, 90)) if valid.size else 0.0,
    }


# 6 tiles -> 6 workers running in parallel, each with 2 CPUs and 4 GB RAM
results = remote_parallel_map(compute_ndvi, TILES, func_cpu=2, func_ram=4, grow=True)
print(f"processed {len(results)} tiles")

### What just happened?

You just read two bands from 6 multi-gigabyte Sentinel-2 tiles — spread across 4 continents — and computed NDVI on each, in parallel. The workers pulled only the byte ranges they needed thanks to COG range reads, so total egress was maybe 50 MB.

GDAL was never installed on your laptop. You wrote a function that uses `rasterio` and handed it to Burla. Burla installed `rasterio` (which bundles GDAL+PROJ) on every worker automatically.

### Inspect the results

NDVI by region — rainforest should be highest (dense vegetation, ~0.8), alpine or urban scenes lower (~0.1–0.4). This is real data, so numbers depend on cloud cover and season.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(results).sort_values("mean_ndvi", ascending=False).reset_index(drop=True)
print(df[["region", "mean_ndvi", "p90_ndvi", "pixels"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(df["region"], df["mean_ndvi"])
ax.set_xlabel("mean NDVI (higher = more vegetation)")
ax.set_title("NDVI by region, Sentinel-2, Aug 2023")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Try this next

- Add more tiles: `sentinel-cogs` has every Sentinel-2 L2A scene since 2017.
- Swap `compute_ndvi` for `reproject_to_4326`, `mask_clouds`, or `chip_to_tiles`.
- Uncap the read size: drop `out_shape=(1024, 1024)` for full-resolution NDVI per tile.
- Write outputs to an S3 bucket inside the worker with `boto3`.

## Thank you for trying Burla! ❤️

### Run the full 2,000-tile version

See [`main.py`](https://github.com/Burla-Cloud/gdal-raster-processing/blob/main/main.py) in this repo — 2,000 tiles across 2,000 workers, writing NDVI GeoTIFFs back to S3.

### Run this on your laptop 🧑‍💻

1. `pip install burla`
2. `burla login`
3. Run the same example code from above ☝️

Congrats! Your laptop now effectively has 1000+ CPUs, and 4000G of RAM &nbsp; : )

<br/>

### Any way we can help? Lets chat!

Schedule a meeting: https://cal.com/jakez  
Send me an email: jake@burla.dev